In [ ]:
import pandas as pd
import numpy as np
import os
import time
from sklearn.utils.class_weight import compute_class_weight

import cudf
import dask_cudf
import dask.array as da

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report, accuracy_score, balanced_accuracy_score,
    f1_score, precision_score, recall_score, matthews_corrcoef,
    cohen_kappa_score, confusion_matrix
)

from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [ ]:
import random
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [ ]:
# ===================== DATASET =====================
class HybridDataset(Dataset):
    def __init__(self, X_seq, X_static, y):
        self.X_seq = torch.FloatTensor(X_seq)
        self.X_static = torch.FloatTensor(X_static)
        self.y = torch.LongTensor(y)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X_seq[idx], self.X_static[idx], self.y[idx]

# ===================== MODEL =====================
class HybridRNNModel(nn.Module):
    def __init__(self, input_dim_per_phase, static_dim):
        super().__init__()
        self.hidden_dim = 128
        self.num_layers = 1
        self.dropout_p = 0.3
        self.num_classes = 3

        #  RNN
        self.rnn = nn.RNN(
            input_size=input_dim_per_phase,
            hidden_size=self.hidden_dim,
            num_layers=self.num_layers,
            batch_first=True
        )

        self.dropout = nn.Dropout(self.dropout_p)

        self.mlp = nn.Sequential(
            nn.Linear(static_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU()
        )

        self.fc = nn.Linear(self.hidden_dim + 32, self.num_classes)

    def forward(self, x_seq, x_static):
        # RNN chỉ trả về output và h_n
        _, h_n = self.rnn(x_seq)

        # Lấy last hidden state
        last_hidden = self.dropout(h_n[-1])

        static_out = self.mlp(x_static)

        combined = torch.cat([last_hidden, static_out], dim=1)
        return self.fc(combined)

In [ ]:
# ===================== METRICS =====================
def gmean_score(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    per_class = []
    for i in range(cm.shape[0]):
        tp = cm[i,i]
        fn = cm[i].sum() - tp
        fp = cm[:,i].sum() - tp
        tn = cm.sum() - tp - fn - fp
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0
        per_class.append(np.sqrt(sens * spec))
    return np.prod(per_class) ** (1/len(per_class)) if per_class else 0


def gmean_per_class(y_true, y_pred, target_class):
    cm = confusion_matrix(y_true, y_pred)
    i = target_class
    tp = cm[i,i]
    fn = cm[i].sum() - tp
    fp = cm[:,i].sum() - tp
    tn = cm.sum() - tp - fn - fp
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    return np.sqrt(recall * specificity)

# ===================== PRINT RESULTS =====================
def print_results(version_name, phase, y_true, y_pred, time_build_model=None, time_predict=None):
    target_names = ['Excellent', 'Good', 'Average']

    print(f"\n{'='*30} {version_name} - Phase {phase} {'='*30}")
    print(classification_report(y_true, y_pred, digits=10, target_names=target_names))

    prec = precision_score(y_true, y_pred, average=None)
    rec = recall_score(y_true, y_pred, average=None)
    f1 = f1_score(y_true, y_pred, average=None)

    gmeans = [gmean_per_class(y_true, y_pred, i) for i in range(3)]

    print("G-Mean per class (one-vs-rest):")
    for i, name in enumerate(target_names):
        print(f"  {name:<10}: {gmeans[i]:.10f}")

    metrics = {
        'Version': version_name,
        'Phase': phase,
        'TimeBuildModel': time_build_model,
        'TimePredict': time_predict,
        'Accuracy': accuracy_score(y_true, y_pred),
        'BalancedAcc': balanced_accuracy_score(y_true, y_pred),
        'Precision Macro': precision_score(y_true, y_pred, average='macro'),
        'Precision Weighted': precision_score(y_true, y_pred, average='weighted'),
        'Recall Macro': recall_score(y_true, y_pred, average='macro'),
        'Recall Weighted': recall_score(y_true, y_pred, average='weighted'),
        'F1-Score Macro': f1_score(y_true, y_pred, average='macro'),
        'F1-Score Weighted': f1_score(y_true, y_pred, average='weighted'),
        'GMean': gmean_score(y_true, y_pred),
        'MCC': matthews_corrcoef(y_true, y_pred),
        'Kappa': cohen_kappa_score(y_true, y_pred),
    }

    for i, name in enumerate(target_names):
        metrics[f'Precision_{name}'] = prec[i]
        metrics[f'Recall_{name}'] = rec[i]
        metrics[f'F1-Score_{name}'] = f1[i]
        metrics[f'G-Mean_{name}'] = gmeans[i]

    for k, v in metrics.items():
        if k not in ['Version','Phase'] and v is not None:
            print(f"{k:22} : {v:.10f}")

    return metrics


# Hàm chuẩn bị dữ liệu và train

In [ ]:
# ===================== TRAIN =====================
def prepare_and_train_hybrid(train_path, val_path, device, version_name):

    print(f"Loading train (GPU): {train_path}")
    # Load tập train bằng dask_cudf
    ddf_train = dask_cudf.read_parquet(train_path)

    # Load tập valid bằng pandas (RAM)
    df_val = pd.read_parquet(val_path, engine='pyarrow') if val_path else None

    # Đếm số lượng mẫu (Dask cần .compute() hoặc len() trên dask_cudf)
    train_len = len(ddf_train)
    print(f"Train samples: {train_len}")
    if df_val is not None:
        print(f"Validation samples: {len(df_val)}")

    # Loại bỏ cột bằng dask_cudf
    cols_to_drop = ['user_id', 'course_id']
    ddf_train = ddf_train.drop(columns=[c for c in cols_to_drop if c in ddf_train.columns])

    if df_val is not None:
        df_val = df_val.drop(columns=cols_to_drop, errors='ignore')

    # Tách y và X cho tập Train
    # Chuyển về numpy để đưa vào PyTorch Dataset (Dask -> CuDF -> Numpy)
    y_train = ddf_train['label_3'].compute().to_numpy()
    X_train_ddf = ddf_train.drop('label_3', axis=1)

    if df_val is not None:
        y_val = df_val['label_3'].values
        X_val_df = df_val.drop('label_3', axis=1)

    # Xác định các cột (Dùng columns của dask_cudf)
    train_columns = X_train_ddf.columns.tolist()
    phase_cols = [c for c in train_columns if any(f"_p{p}_" in c for p in ['1','2','3','4'])]
    static_cols = [c for c in train_columns if c not in phase_cols]

    for p in ['1','2','3','4']:
        print(f"Phase {p}: {len([c for c in phase_cols if f'_p{p}_' in c])} features")

    # Hàm build_seq xử lý trên Dask/GPU
    def build_seq_dask(ddf, p_cols):
        phases = []
        for p in ['1','2','3','4']:
            cols = sorted([c for c in p_cols if f"_p{p}_" in c])
            # Chuyển từng phase về numpy
            phases.append(ddf[cols].compute().to_numpy())
        # Stack lại thành (N, T, F)
        return np.stack(phases, axis=1)

    # Hàm build_seq cho Pandas (Valid)
    def build_seq_pandas(df, p_cols):
        phases = []
        for p in ['1','2','3','4']:
            cols = sorted([c for c in p_cols if f"_p{p}_" in c])
            phases.append(df[cols].values)
        return np.stack(phases, axis=1)

    # Thực hiện build sequence
    X_seq_train = build_seq_dask(X_train_ddf, phase_cols)
    X_static_train = X_train_ddf[static_cols].compute().to_numpy()

    print(f"Time-series shape: {X_seq_train.shape}")
    print(f"Static feature shape: {X_static_train.shape}")

    # ---Tiền xử lý & Train---
    scaler_seq = StandardScaler()
    N, T, F = X_seq_train.shape
    X_seq_train = scaler_seq.fit_transform(X_seq_train.reshape(-1, F)).reshape(N, T, F)

    scaler_static = StandardScaler()
    X_static_train = scaler_static.fit_transform(X_static_train)

    if df_val is not None:
        X_seq_val = build_seq_pandas(X_val_df, phase_cols)
        X_static_val = X_val_df[static_cols].values
        N2 = X_seq_val.shape[0]
        X_seq_val = scaler_seq.transform(X_seq_val.reshape(-1, F)).reshape(N2, T, F)
        X_static_val = scaler_static.transform(X_static_val)

    le = LabelEncoder()
    y_train_enc = le.fit_transform(y_train)
    print(f"Classes: {le.classes_}")

    if df_val is not None:
        y_val_enc = le.transform(y_val)

    train_loader = DataLoader(HybridDataset(X_seq_train, X_static_train, y_train_enc), batch_size=256, shuffle=True)

    if df_val is not None:
        val_loader = DataLoader(HybridDataset(X_seq_val, X_static_val, y_val_enc), batch_size=256, shuffle=False)

    model = HybridRNNModel(F, X_static_train.shape[1]).to(device)

    # Class weights
    unique_classes = np.unique(y_train_enc)
    weights = compute_class_weight('balanced', classes=unique_classes, y=y_train_enc)
    weights = torch.FloatTensor(weights).to(device)

    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # Vòng lặp Training
    best_loss = float('inf')
    patience = 10
    wait = 0
    start_train = time.perf_counter()

    for epoch in range(50):
        model.train()
        train_loss = 0
        for xb_seq, xb_static, yb in train_loader:
            xb_seq, xb_static, yb = xb_seq.to(device), xb_static.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(xb_seq, xb_static)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)

        if df_val is not None:
            model.eval()
            val_loss = 0
            with torch.no_grad():
                for xb_seq, xb_static, yb in val_loader:
                    xb_seq, xb_static, yb = xb_seq.to(device), xb_static.to(device), yb.to(device)
                    val_loss += criterion(model(xb_seq, xb_static), yb).item()
            val_loss /= len(val_loader)
            print(f"Epoch {epoch+1}: train = {train_loss:.4f}, val = {val_loss:.4f}")
            monitor = val_loss
        else:
            print(f"Epoch {epoch+1}: loss = {train_loss:.4f}")
            monitor = train_loss

        if monitor < best_loss - 1e-4:
            best_loss = monitor
            best_state = model.state_dict()
            wait = 0
        else:
            wait += 1
            if wait >= patience: break

    model.load_state_dict(best_state)
    time_build = time.perf_counter() - start_train

    os.makedirs("saved_models", exist_ok=True)
    torch.save(model.state_dict(), f"saved_models/RNN_{version_name}.pt")

    return model, scaler_seq, scaler_static, le, phase_cols, static_cols, time_build

# Chạy từng version

In [ ]:
def run_experiment(base_path, train_file, val_file, test_prefix, version_name, device=None):
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    print(f"\n{'#'*20}")
    print(f"Version: {version_name}")
    print(f"{'#'*20}")

    # 1. Huấn luyện mô hình
    model, scaler_seq, scaler_static, le, phase_cols, static_cols, time_build = \
        prepare_and_train_hybrid(f"{base_path_1}/{train_file}", f"{base_path}/{val_file}", device, version_name)

    results = []

    # Chuyển model sang chế độ eval một lần duy nhất trước khi test
    model.eval()

    # 2. Vòng lặp test qua 4 phase
    for phase in range(1, 5):
        test_path = f"{base_path}/{test_prefix}_{phase}.parquet"
        print(f"\n--- Test Phase {phase}: {test_path} ---")

        # Load tập test bằng Pandas
        df = pd.read_parquet(test_path, engine='pyarrow')
        df = df.drop(columns=['user_id','course_id'], errors='ignore')

        y_test_raw = df['label_3'].values
        X_df = df.drop('label_3', axis=1)

        # Hàm build sequence (Giống như cũ nhưng xử lý gọn hơn)
        def build_seq_local(df_input):
            phases_list = []
            for p in ['1','2','3','4']:
                cols = sorted([c for c in phase_cols if f"_p{p}_" in c])
                phases_list.append(df_input[cols].values)
            return np.stack(phases_list, axis=1)

        X_seq_test = build_seq_local(X_df)
        X_static_test = X_df[static_cols].values

        # Transform dữ liệu
        N, T, F = X_seq_test.shape
        X_seq_test = scaler_seq.transform(X_seq_test.reshape(-1, F)).reshape(N, T, F)
        X_static_test = scaler_static.transform(X_static_test)

        # Mã hóa nhãn thực tế
        y_test_enc = le.transform(y_test_raw)

        # 3. CHIA BATCH CHO INFERENCE (Khắc phục lỗi OOM)
        test_dataset = HybridDataset(X_seq_test, X_static_test, y_test_enc)
        test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

        all_probs = []
        start_time = time.perf_counter()

        with torch.no_grad():
            for xb_seq, xb_static, _ in test_loader:
                # Đưa từng batch nhỏ lên GPU
                xb_seq = xb_seq.to(device)
                xb_static = xb_static.to(device)

                outputs = model(xb_seq, xb_static)

                # Tính xác suất và đưa về CPU ngay lập tức để giải phóng VRAM
                prob = torch.softmax(outputs, dim=1).cpu().numpy()
                all_probs.append(prob)

        time_pred = time.perf_counter() - start_time

        # Gộp các batch lại thành mảng lớn trên CPU/RAM
        probs = np.vstack(all_probs)
        preds = np.argmax(probs, axis=1)

        # 4. Tính toán và in kết quả
        metrics = print_results(version_name, phase, y_test_enc, preds, time_build, time_pred)

        # 5. Lưu kết quả
        os.makedirs("results_RNN", exist_ok=True)

        # Lưu Confusion Matrix
        cm = confusion_matrix(y_test_enc, preds)
        pd.DataFrame(cm).to_csv(f"results_RNN/confusion_matrix_{version_name}_phase{phase}.csv", index=False)

        # Lưu Ma trận xác suất
        df_prob = pd.DataFrame(probs, columns=[f"Prob_Class_{i}" for i in range(probs.shape[1])])
        df_prob['y_true'] = y_test_enc
        df_prob['y_pred'] = preds
        df_prob.to_csv(f"results_RNN/probability_matrix_{version_name}_phase{phase}.csv", index=False)

        results.append(metrics)

        # Giải phóng bộ nhớ tạm sau mỗi phase test
        del df, X_seq_test, X_static_test, all_probs, probs, preds
        torch.cuda.empty_cache()

    # Tổng hợp bảng kết quả cuối cùng
    df_results = pd.DataFrame(results).round(10)

    ordered_cols = [
        "Version","Phase","TimeBuildModel","TimePredict","Accuracy","BalancedAcc",
        "Precision Macro","Precision Weighted","Recall Macro","Recall Weighted",
        "F1-Score Macro","F1-Score Weighted","GMean","MCC","Kappa",
        "Precision_Excellent","Recall_Excellent","F1-Score_Excellent","G-Mean_Excellent",
        "Precision_Good","Recall_Good","F1-Score_Good","G-Mean_Good",
        "Precision_Average","Recall_Average","F1-Score_Average","G-Mean_Average"
    ]

    df_results = df_results[[c for c in ordered_cols if c in df_results.columns]]

    return df_results

## V_Median

In [ ]:
base_path = "/kaggle/input/datasets/anhtran10/lo-dataset/Median/Median"

In [ ]:
df_v1 = run_experiment(
    base_path=base_path,
    train_file="train_median.parquet",
    val_file="val.parquet",
    test_prefix="test",
    version_name="V1 (Median)"
)
df_v1


####################
Version: V1 (Median)
####################
Loading train (GPU): /kaggle/input/datasets/anhtran10/lo-dataset/Median/Median/train_median.parquet
Train samples: 1859619
Validation samples: 232452
Phase 1: 39 features
Phase 2: 39 features
Phase 3: 39 features
Phase 4: 39 features
Time-series shape: (1859619, 4, 39)
Static feature shape: (1859619, 23)
Classes: [0 1 2]
Epoch 1: train = 0.2869, val = 0.1921
Epoch 2: train = 0.1987, val = 0.1412
Epoch 3: train = 0.1691, val = 0.1239
Epoch 4: train = 0.1639, val = 0.1175
Epoch 5: train = 0.1519, val = 0.1115
Epoch 6: train = 0.1496, val = 0.1697
Epoch 7: train = 0.1437, val = 0.1157
Epoch 8: train = 0.1351, val = 0.1081
Epoch 9: train = 0.1377, val = 0.1202
Epoch 10: train = 0.1382, val = 0.1033
Epoch 11: train = 0.1370, val = 0.1141
Epoch 12: train = 0.1312, val = 0.1071
Epoch 13: train = 0.1296, val = 0.1227
Epoch 14: train = 0.1293, val = 0.1131
Epoch 15: train = 0.1259, val = 0.1127
Epoch 16: train = 0.1240, val = 0.142

,Version,Phase,TimeBuildModel,TimePredict,Accuracy,BalancedAcc,Precision Macro,Precision Weighted,Recall Macro,Recall Weighted,F1-Score Macro,F1-Score Weighted,GMean,MCC,Kappa,Precision_Excellent,Recall_Excellent,F1-Score_Excellent,G-Mean_Excellent,Precision_Good,Recall_Good,F1-Score_Good,G-Mean_Good,Precision_Average,Recall_Average,F1-Score_Average,G-Mean_Average
0,V1 (Median),1,731.163732,2.668746,0.584746,0.526976,0.335395,0.996432,0.526976,0.584746,0.249964,0.734958,0.000000,0.060683,0.008518,0.000000,0.000000,0.000000,0.000000,0.006208,0.996694,0.012340,0.762723,0.999978,0.584233,0.737554,0.762965
1,V1 (Median),2,731.163732,2.455044,0.719965,0.566165,0.336308,0.996372,0.566165,0.719965,0.285010,0.834231,0.000000,0.080177,0.015132,0.000000,0.000000,0.000000,0.000000,0.009014,0.978512,0.017864,0.838953,0.999910,0.719983,0.837166,0.840797
2,V1 (Median),3,731.163732,2.635296,0.915312,0.634978,0.422887,0.996626,0.634978,0.915312,0.353467,0.952916,0.525439,0.162049,0.060433,0.240000,0.026906,0.048387,0.164023,0.028778,0.961983,0.055884,0.938342,0.999882,0.916045,0.956130,0.942543
3,V1 (Median),4,731.163732,2.438428,0.987821,0.933625,0.501680,0.997134,0.933625,0.987821,0.591415,0.991677,0.959024,0.452784,0.352919,0.313099,0.878924,0.461720,0.936641,0.192046,0.933884,0.318579,0.961410,0.999895,0.988067,0.993946,0.979504


In [ ]:
df_v1.to_csv("results_v1.csv", index=False)

## V_SMOTE

In [ ]:
base_path_1 = "/kaggle/input/datasets/uyentran10/lo-smote-test"

In [ ]:
base_path = "/kaggle/input/datasets/anhtran10/lo-dataset/Median/Median"

In [ ]:
df_v22 = run_experiment(
    base_path=base_path,
    train_file="train_median_smote.parquet",
    val_file="val.parquet",
    test_prefix="test",
    version_name="V22 (Median SMOTE)"
)
df_v22


####################
Version: V22 (Median SMOTE)
####################
Loading train (GPU): /kaggle/input/datasets/uyentran10/lo-smote-test/train_median_smote.parquet
Train samples: 5558991
Validation samples: 232452
Phase 1: 39 features
Phase 2: 39 features
Phase 3: 39 features
Phase 4: 39 features
Time-series shape: (5558991, 4, 39)
Static feature shape: (5558991, 23)
Classes: [0 1 2]
Epoch 1: train = 0.0636, val = 0.0297
Epoch 2: train = 0.0337, val = 0.0319
Epoch 3: train = 0.0291, val = 0.0347
Epoch 4: train = 0.0268, val = 0.0329
Epoch 5: train = 0.0252, val = 0.0288
Epoch 6: train = 0.0241, val = 0.0353
Epoch 7: train = 0.0232, val = 0.0263
Epoch 8: train = 0.0226, val = 0.0265
Epoch 9: train = 0.0221, val = 0.0256
Epoch 10: train = 0.0217, val = 0.0257
Epoch 11: train = 0.0213, val = 0.0239
Epoch 12: train = 0.0209, val = 0.0237
Epoch 13: train = 0.0207, val = 0.0316
Epoch 14: train = 0.0204, val = 0.0246
Epoch 15: train = 0.0202, val = 0.0250
Epoch 16: train = 0.0199, val = 0.

,Version,Phase,TimeBuildModel,TimePredict,Accuracy,BalancedAcc,Precision Macro,Precision Weighted,Recall Macro,Recall Weighted,F1-Score Macro,F1-Score Weighted,GMean,MCC,Kappa,Precision_Excellent,Recall_Excellent,F1-Score_Excellent,G-Mean_Excellent,Precision_Good,Recall_Good,F1-Score_Good,G-Mean_Good,Precision_Average,Recall_Average,F1-Score_Average,G-Mean_Average
0,V22 (Median SMOTE),1,5052.046005,2.731062,0.988071,0.590967,0.433593,0.994948,0.590967,0.988071,0.475709,0.991295,0.634440,0.224148,0.191237,0.220000,0.493274,0.304288,0.701744,0.082703,0.289256,0.128629,0.535569,0.998077,0.990372,0.994210,0.679482
1,V22 (Median SMOTE),2,5052.046005,2.965358,0.985447,0.608320,0.421143,0.994970,0.608320,0.985447,0.464153,0.989916,0.652213,0.210841,0.168978,0.197044,0.538117,0.288462,0.732792,0.068225,0.299174,0.111111,0.544044,0.998159,0.987670,0.992887,0.695911
2,V22 (Median SMOTE),3,5052.046005,2.701321,0.968561,0.738466,0.415586,0.995750,0.738466,0.968561,0.461128,0.981264,0.799484,0.211892,0.124960,0.198718,0.695067,0.309073,0.832584,0.049050,0.550413,0.090073,0.731496,0.998991,0.969917,0.984239,0.839053
3,V22 (Median SMOTE),4,5052.046005,3.022269,0.993930,0.929636,0.568402,0.997478,0.929636,0.993930,0.669925,0.995279,0.952891,0.573563,0.516117,0.355517,0.910314,0.511335,0.953347,0.349902,0.884298,0.501406,0.938353,0.999787,0.994297,0.997034,0.967190


In [ ]:
df_v22.to_csv("results_v22.csv", index=False)

## V_GAN

In [ ]:
base_path_1 = "/kaggle/input/datasets/anhtran10/lo-gan-test"

In [ ]:
df_v23 = run_experiment(
    base_path=base_path,
    train_file="train_median_gan.parquet",
    val_file="val.parquet",
    test_prefix="test",
    version_name="V23 (Median GAN)"
)
df_v23


####################
Version: V23 (Median GAN)
####################
Loading train (GPU): /kaggle/input/datasets/anhtran10/lo-gan-test/train_median_gan.parquet
Train samples: 5558991
Validation samples: 232452
Phase 1: 39 features
Phase 2: 39 features
Phase 3: 39 features
Phase 4: 39 features
Time-series shape: (5558991, 4, 39)
Static feature shape: (5558991, 23)
Classes: [0 1 2]
Epoch 1: train = 0.0037, val = 0.0069
Epoch 2: train = 0.0025, val = 0.0062
Epoch 3: train = 0.0023, val = 0.0057
Epoch 4: train = 0.0022, val = 0.0061
Epoch 5: train = 0.0022, val = 0.0062
Epoch 6: train = 0.0022, val = 0.0058
Epoch 7: train = 0.0021, val = 0.0061
Epoch 8: train = 0.0021, val = 0.0054
Epoch 9: train = 0.0021, val = 0.0063
Epoch 10: train = 0.0021, val = 0.0056
Epoch 11: train = 0.0021, val = 0.0053
Epoch 12: train = 0.0021, val = 0.0055
Epoch 13: train = 0.0021, val = 0.0054
Epoch 14: train = 0.0021, val = 0.0056
Epoch 15: train = 0.0021, val = 0.0053
Epoch 16: train = 0.0021, val = 0.0059
Ep

,Version,Phase,TimeBuildModel,TimePredict,Accuracy,BalancedAcc,Precision Macro,Precision Weighted,Recall Macro,Recall Weighted,F1-Score Macro,F1-Score Weighted,GMean,MCC,Kappa,Precision_Excellent,Recall_Excellent,F1-Score_Excellent,G-Mean_Excellent,Precision_Good,Recall_Good,F1-Score_Good,G-Mean_Good,Precision_Average,Recall_Average,F1-Score_Average,G-Mean_Average
0,V23 (Median GAN),1,3155.788603,2.913176,0.926097,0.791746,0.365449,0.995969,0.791746,0.926097,0.380756,0.958655,0.848248,0.156694,0.062980,0.071903,0.825112,0.132279,0.903700,0.025048,0.623140,0.048160,0.764003,0.999395,0.926985,0.961829,0.883993
1,V23 (Median GAN),2,3155.788603,2.683588,0.926424,0.797195,0.365716,0.995994,0.797195,0.926424,0.381312,0.958827,0.852727,0.158773,0.063929,0.072097,0.829596,0.132664,0.906141,0.025632,0.634711,0.049275,0.771200,0.999418,0.927279,0.961998,0.887294
2,V23 (Median GAN),3,3155.788603,2.851577,0.974115,0.861545,0.395256,0.996357,0.861545,0.974115,0.439780,0.984181,0.900237,0.286365,0.179529,0.086337,0.923767,0.157915,0.956606,0.099856,0.685950,0.174333,0.821512,0.999575,0.974916,0.987092,0.928376
3,V23 (Median GAN),4,3155.788603,2.674820,0.998167,0.792549,0.845634,0.997997,0.792549,0.998167,0.811998,0.998029,0.819413,0.718422,0.714150,0.780992,0.847534,0.812903,0.920511,0.757075,0.530579,0.623907,0.728246,0.998835,0.999534,0.999184,0.820731


In [ ]:
df_v23.to_csv("results_v23.csv", index=False)

## V_CDSMOTE

In [ ]:
df_v2 = run_experiment(
    base_path=base_path,
    train_file="train_median_cdsmote.parquet",
    val_file="val.parquet",
    test_prefix="test",
    version_name="V2 (Median CDS)"
)
df_v2


####################
Version: V2 (Median CDS)
####################
Loading train (GPU): /kaggle/input/datasets/anhtran10/lo-dataset/Median/Median/train_median_cdsmote.parquet
Train samples: 5558987
Validation samples: 232452
Phase 1: 39 features
Phase 2: 39 features
Phase 3: 39 features
Phase 4: 39 features
Time-series shape: (5558987, 4, 39)
Static feature shape: (5558987, 23)
Classes: [0 1 2]
Epoch 1: train = 0.0481, val = 0.0333
Epoch 2: train = 0.0255, val = 0.0239
Epoch 3: train = 0.0221, val = 0.0274
Epoch 4: train = 0.0202, val = 0.0201
Epoch 5: train = 0.0191, val = 0.0241
Epoch 6: train = 0.0183, val = 0.0283
Epoch 7: train = 0.0175, val = 0.0236
Epoch 8: train = 0.0171, val = 0.0236
Epoch 9: train = 0.0168, val = 0.0228
Epoch 10: train = 0.0164, val = 0.0234
Epoch 11: train = 0.0161, val = 0.0236
Epoch 12: train = 0.0159, val = 0.0250
Epoch 13: train = 0.0156, val = 0.0259
Epoch 14: train = 0.0155, val = 0.0207

--- Test Phase 1: /kaggle/input/datasets/anhtran10/lo-dataset/M

,Version,Phase,TimeBuildModel,TimePredict,Accuracy,BalancedAcc,Precision Macro,Precision Weighted,Recall Macro,Recall Weighted,F1-Score Macro,F1-Score Weighted,GMean,MCC,Kappa,Precision_Excellent,Recall_Excellent,F1-Score_Excellent,G-Mean_Excellent,Precision_Good,Recall_Good,F1-Score_Good,G-Mean_Good,Precision_Average,Recall_Average,F1-Score_Average,G-Mean_Average
0,V2 (Median CDS),1,1501.551751,2.446448,0.987632,0.465499,0.392773,0.994854,0.465499,0.987632,0.389766,0.990960,0.384016,0.206939,0.175410,0.090909,0.017937,0.029963,0.133918,0.089320,0.388430,0.145241,0.620013,0.998089,0.990131,0.994094,0.682034
1,V2 (Median CDS),2,1501.551751,2.661302,0.986720,0.511475,0.438994,0.995154,0.511475,0.986720,0.434367,0.990621,0.546490,0.222328,0.183095,0.229508,0.125561,0.162319,0.354274,0.089217,0.419835,0.147161,0.644313,0.998257,0.989030,0.993622,0.715009
2,V2 (Median CDS),3,1501.551751,2.420926,0.978710,0.668722,0.488881,0.996097,0.668722,0.978710,0.502431,0.986718,0.746118,0.260062,0.177519,0.387560,0.363229,0.375000,0.602518,0.080008,0.662810,0.142781,0.805995,0.999076,0.980127,0.989511,0.855304
3,V2 (Median CDS),4,1501.551751,2.700297,0.995027,0.925678,0.606605,0.997653,0.925678,0.995027,0.706711,0.996003,0.950971,0.611998,0.565641,0.430736,0.892377,0.581022,0.944122,0.389291,0.889256,0.541520,0.941286,0.999788,0.995402,0.997590,0.967727


In [ ]:
df_v2.to_csv("results_v2.csv", index=False)

## V_SMOTified GAN

In [ ]:
base_path_1 = "/kaggle/input/datasets/uyentran10/lo-smotifiedgan-test"

In [ ]:
df_v24 = run_experiment(
    base_path=base_path,
    train_file="train_median_smotified_gan.parquet",
    val_file="val.parquet",
    test_prefix="test",
    version_name="V24 (Median SMOTified GAN)"
)
df_v24


####################
Version: V24 (Median SMOTified GAN)
####################
Loading train (GPU): /kaggle/input/datasets/uyentran10/lo-smotifiedgan-test/train_median_smotified_gan.parquet
Train samples: 5558991
Validation samples: 232452
Phase 1: 39 features
Phase 2: 39 features
Phase 3: 39 features
Phase 4: 39 features
Time-series shape: (5558991, 4, 39)
Static feature shape: (5558991, 23)
Classes: [0 1 2]
Epoch 1: train = 0.0038, val = 0.0064
Epoch 2: train = 0.0025, val = 0.0061
Epoch 3: train = 0.0023, val = 0.0056
Epoch 4: train = 0.0023, val = 0.0056
Epoch 5: train = 0.0022, val = 0.0057
Epoch 6: train = 0.0021, val = 0.0054
Epoch 7: train = 0.0022, val = 0.0052
Epoch 8: train = 0.0021, val = 0.0054
Epoch 9: train = 0.0021, val = 0.0054
Epoch 10: train = 0.0021, val = 0.0055
Epoch 11: train = 0.0021, val = 0.0055
Epoch 12: train = 0.0021, val = 0.0052
Epoch 13: train = 0.0021, val = 0.0052
Epoch 14: train = 0.0020, val = 0.0054
Epoch 15: train = 0.0020, val = 0.0056
Epoch 16: t

,Version,Phase,TimeBuildModel,TimePredict,Accuracy,BalancedAcc,Precision Macro,Precision Weighted,Recall Macro,Recall Weighted,F1-Score Macro,F1-Score Weighted,GMean,MCC,Kappa,Precision_Excellent,Recall_Excellent,F1-Score_Excellent,G-Mean_Excellent,Precision_Good,Recall_Good,F1-Score_Good,G-Mean_Good,Precision_Average,Recall_Average,F1-Score_Average,G-Mean_Average
0,V24 (Median SMOTified GAN),1,2126.52977,2.954786,0.984440,0.457359,0.355474,0.994697,0.457359,0.984440,0.369525,0.989250,0.000000,0.179840,0.142075,0.000000,0.000000,0.000000,0.000000,0.068348,0.385124,0.116094,0.616318,0.998075,0.986953,0.992483,0.679186
1,V24 (Median SMOTified GAN),2,2126.52977,2.716229,0.988677,0.486253,0.369902,0.995111,0.486253,0.988677,0.391508,0.991595,0.000000,0.250676,0.215342,0.000000,0.000000,0.000000,0.000000,0.111330,0.467769,0.179854,0.680596,0.998378,0.990990,0.994670,0.737947
2,V24 (Median SMOTified GAN),3,2126.52977,3.002930,0.995917,0.457209,0.770559,0.996052,0.457209,0.995917,0.464187,0.995553,0.409740,0.362046,0.359630,1.000000,0.035874,0.069264,0.189405,0.313846,0.337190,0.325100,0.580122,0.997830,0.998562,0.998196,0.626057
3,V24 (Median SMOTified GAN),4,2126.52977,2.762723,0.998288,0.813256,0.862042,0.998147,0.813256,0.998288,0.827954,0.998148,0.831988,0.736977,0.732407,0.788235,0.901345,0.841004,0.949282,0.799020,0.538843,0.643633,0.733929,0.998870,0.999581,0.999225,0.826613


In [ ]:
df_v24.to_csv("results_v24.csv", index=False)

## V_CDSGAN

In [ ]:
base_path_1 = "/kaggle/input/datasets/hngliththu/cdsmote-gan"

In [ ]:
base_path = "/kaggle/input/datasets/anhtran10/lo-dataset/Median/Median"

In [ ]:
df_v13 = run_experiment(
    base_path=base_path,
    train_file="train_median_cdsmote_gan.parquet",
    val_file="val.parquet",
    test_prefix="test",
    version_name="V13 (Median CDSMOTified GAN)"
)
df_v13


####################
Version: V13 (Median CDSMOTified GAN)
####################
Loading train (GPU): /kaggle/input/datasets/hngliththu/cdsmote-gan/train_median_cdsmote_gan.parquet
Train samples: 5558987
Validation samples: 232452
Phase 1: 39 features
Phase 2: 39 features
Phase 3: 39 features
Phase 4: 39 features
Time-series shape: (5558987, 4, 39)
Static feature shape: (5558987, 23)
Classes: [0 1 2]
Epoch 1: train = 0.0039, val = 0.0069
Epoch 2: train = 0.0025, val = 0.0058
Epoch 3: train = 0.0024, val = 0.0058
Epoch 4: train = 0.0023, val = 0.0057
Epoch 5: train = 0.0022, val = 0.0058
Epoch 6: train = 0.0022, val = 0.0054
Epoch 7: train = 0.0022, val = 0.0057
Epoch 8: train = 0.0022, val = 0.0055
Epoch 9: train = 0.0022, val = 0.0053
Epoch 10: train = 0.0021, val = 0.0056
Epoch 11: train = 0.0021, val = 0.0052
Epoch 12: train = 0.0021, val = 0.0056
Epoch 13: train = 0.0021, val = 0.0052
Epoch 14: train = 0.0021, val = 0.0060
Epoch 15: train = 0.0021, val = 0.0056
Epoch 16: train = 0.

,Version,Phase,TimeBuildModel,TimePredict,Accuracy,BalancedAcc,Precision Macro,Precision Weighted,Recall Macro,Recall Weighted,F1-Score Macro,F1-Score Weighted,GMean,MCC,Kappa,Precision_Excellent,Recall_Excellent,F1-Score_Excellent,G-Mean_Excellent,Precision_Good,Recall_Good,F1-Score_Good,G-Mean_Good,Precision_Average,Recall_Average,F1-Score_Average,G-Mean_Average
0,V13 (Median CDSMOTified GAN),1,2224.841093,2.633023,0.990329,0.452784,0.426761,0.994200,0.452784,0.990329,0.425347,0.992160,0.444083,0.154514,0.143324,0.195804,0.125561,0.153005,0.354257,0.087139,0.239669,0.127810,0.487954,0.997338,0.993123,0.995226,0.506633
1,V13 (Median CDSMOTified GAN),2,2224.841093,2.397889,0.990454,0.449134,0.432459,0.994202,0.449134,0.990454,0.425137,0.992223,0.438334,0.152875,0.142348,0.214286,0.121076,0.154728,0.347886,0.085766,0.233058,0.125389,0.481193,0.997325,0.993269,0.995293,0.503106
2,V13 (Median CDSMOTified GAN),3,2224.841093,2.622143,0.996571,0.536493,0.707192,0.995481,0.536493,0.996571,0.593127,0.995851,0.505379,0.359474,0.331362,0.737589,0.466368,0.571429,0.682857,0.386667,0.143802,0.209639,0.379099,0.997320,0.999309,0.998314,0.498619
3,V13 (Median CDSMOTified GAN),4,2224.841093,2.426257,0.998318,0.794893,0.890477,0.998186,0.794893,0.998318,0.825071,0.998117,0.809650,0.734217,0.723823,0.808943,0.892377,0.848614,0.944561,0.863768,0.492562,0.627368,0.701756,0.998719,0.999741,0.999230,0.800711


In [ ]:
df_v13.to_csv("results_v13.csv", index=False)